In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
import getpass

In [5]:
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [11]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENAI_API_KEY,
    model_name="gpt-5-nano"
)

## Split

In [1]:
# the file contains the shorter version of the book, i.e. ~18 000 tokens
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [6]:
text_chunks_chain = (
    RunnableLambda(lambda x: [
        {
            'chunk': text_chunk,
        } for text_chunk in TokenTextSplitter(chunk_size=3_000, chunk_overlap=100).split_text(x)
    ])
)

## Map

In [9]:
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

In [13]:
summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)

In [14]:
summarize_chunk_chain = summarize_chunk_prompt | llm

In [16]:
summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

## Reduce

In [17]:
summarize_summaries_prompt_template = """
Write a concise summary of the following text,
which joins several summaries, and include the main details.
Text: {summaries}
"""

In [19]:
summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)

In [20]:
summarize_reduce_chain = (
    RunnableLambda(lambda x:
                   {
                       'summaries': '\n'.join([i['summary'] for i in x]),
                   })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

## MapReduce combined chain

In [21]:
map_reduce_chain = (
    text_chunks_chain
    | summarize_map_chain.map()
    | summarize_reduce_chain
)

## MapReduce execution

In [22]:
# summary = map_reduce_chain.invoke(moby_dick_book)